# Comparação de modelos

Este caderno compara os três algoritmos avaliados pelo grupo para prever a temperatura de cada componente do fogão num ensaio térmico: a floresta aleatória, o gradiente e a mescla entre os dois. Para cada um, a configuração fixa usada no caderno de modelagem é confrontada com a melhor configuração encontrada por busca aleatória de hiperparâmetros, e a melhor das duas é mantida. Ao final, o melhor algoritmo de cada área térmica é identificado, a escolha é justificada, e o modelo vencedor é explicado.

A busca aleatória segue o método do caderno random_search_13cat, desenvolvido por outra frente do grupo, reexecutado aqui sobre a base corrigida, que passou a incluir as colunas do motivo do ensaio.

A última seção mostra como a formulação efetivamente entregue resulta desta comparação.

## 1. Como executar

No Google Colab, a primeira célula de código pede o envio dos arquivos de dados, que não estão no repositório por compromisso de proteção de dados com a parceira. São nove arquivos: os sete conjuntos por área, com nome dataset seguido da área e do sufixo 13cat, a tabela limites_ativos.csv e a partição particao_treino_teste.json.

Fora do Colab, o caderno procura os arquivos na própria pasta e nas pastas do repositório, e dispensa o envio.

A execução completa levou cerca de onze minutos na máquina em que o caderno foi executado, quase todos consumidos pela busca de hiperparâmetros. No Colab gratuito, que dispõe de menos núcleos de processamento, o tempo tende a ser maior.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import json
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut, RandomizedSearchCV
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

SEMENTE = 42
AREAS = ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling", "mainback", "ipc_comp", "coil_temps"]
ARQUIVOS = [f"dataset_{a}_13cat.csv" for a in AREAS] + ["limites_ativos.csv", "particao_treino_teste.json"]

try:
    from google.colab import files
    faltando = [f for f in ARQUIVOS if not os.path.exists(f)]
    if faltando:
        print("Envie os arquivos:", faltando)
        files.upload()
except ImportError:
    pass


def localiza(nome):
    """Procura o arquivo na pasta atual e nas pastas do repositorio.

    Os CSVs sao lidos apenas do colabs_S4; so a particao JSON pode vir da pasta de preparacao."""
    pastas = (".", "notebooks/colabs_S4")
    if not nome.endswith(".csv"):
        pastas += ("../colabs_S3/preparacao_testes", "notebooks/colabs_S3/preparacao_testes")
    for pasta in pastas:
        caminho = os.path.join(pasta, nome)
        if os.path.exists(caminho):
            return caminho
    raise FileNotFoundError(f"{nome} nao encontrado. Envie o arquivo ou execute a partir do repositorio.")


print("Arquivos localizados:", all(os.path.exists(localiza(f)) for f in ARQUIVOS))

## 2. Os dados e o protocolo de validação

Cada área térmica tem um conjunto próprio, com uma linha por ensaio e uma coluna por componente. O conjunto é convertido para o formato longo, com uma linha por par de ensaio e componente, de modo que um único modelo por área aprenda todos os componentes daquela área.

As variáveis de entrada são os atributos de configuração do ensaio, o perfil de perdas imposto às bobinas e o motivo do ensaio, este codificado em seis categorias mutuamente exclusivas. As colunas de ordem de acionamento das bobinas ficam de fora, por terem sido medidas como prejudiciais no caderno de modelagem.

A validação retira uma configuração experimental inteira por vez, com todos os seus componentes e todos os ensaios que a repetem, treina com as demais e prevê a que ficou de fora. São dezessete dobras. Uma divisão aleatória das linhas colocaria componentes do mesmo ensaio nos dois lados, e o modelo passaria a reconhecer a configuração em vez de generalizar para uma nova; medido nesta base, esse atalho faz o erro parecer quarenta e cinco por cento menor.

Os dez ensaios do conjunto de teste reservado não são lidos neste caderno.

In [ ]:
ENTRADAS = ("blower_direction_", "housing_", "board_position_", "run_case_", "losses_",
            "deflector_hole_", "material_coil_support_", "reason_")
EXCLUIDAS = ("loss_profile_order_",)

particao = json.load(open(localiza("particao_treino_teste.json"), encoding="utf-8"))
DOBRA = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}
TESTE = set(particao["run_idx_teste"])
LIMITES = pd.read_csv(localiza("limites_ativos.csv")).set_index("tc_name")["limite_num"]


def carrega(area):
    """Formato longo da area, apenas com os ensaios de desenvolvimento."""
    d = pd.read_csv(localiza(f"dataset_{area}_13cat.csv"))
    d["run_idx"] = np.arange(len(d))
    entradas = [c for c in d.columns if c.startswith(ENTRADAS)]
    alvos = [c for c in d.columns if c in LIMITES.index]
    sobra = [c for c in d.columns if c not in entradas + alvos + ["run_idx"] and not c.startswith(EXCLUIDAS)]
    assert not sobra, f"{area}: colunas nao classificadas: {sobra}"
    longo = (d[d["run_idx"].map(lambda r: DOBRA.get(r) is not None and r not in TESTE)]
             .melt(id_vars=entradas + ["run_idx"], value_vars=alvos,
                   var_name="componente", value_name="temperatura")
             .dropna(subset=["temperatura"])
             .reset_index(drop=True))
    X = longo[entradas + ["componente"]]
    y = longo["temperatura"].to_numpy()
    grupos = longo["run_idx"].map(DOBRA).astype(int).to_numpy()
    return X, y, grupos


DADOS = {a: carrega(a) for a in AREAS}
for a, (X, y, g) in DADOS.items():
    print(f"{a:<15} {len(y):>5} leituras | {X['componente'].nunique():>2} componentes | "
          f"{len(set(g))} dobras | {X.shape[1] - 1} variaveis de entrada")
print("Total:", sum(len(v[1]) for v in DADOS.values()), "leituras")

## 3. As métricas, e por que estas

O uso pretendido do modelo é uma triagem prévia: antecipar quais componentes provavelmente ultrapassarão o seu limite de segurança, para que o ensaio físico seja dirigido a eles. As métricas foram escolhidas a partir desse uso.

O erro médio absoluto, em graus Celsius, é a métrica principal. É a única expressa na unidade do problema e diretamente comparável à folga de projeto de cada componente, e é ela que decide a escolha entre modelos neste caderno.

Três métricas medem a triagem, sob uma margem de oito graus somada à previsão, escolhida na curva de margens da Seção 4.3 da documentação para que todos os modelos sejam comparados pelo mesmo critério, isto é, um componente é sinalizado quando a previsão somada a oito graus atinge o seu limite. As aprovações indevidas contam os componentes que ultrapassaram o limite e que a triagem deixou passar; é o erro caro, porque deixa passar um risco. As reprovações desnecessárias contam os componentes sinalizados sem necessidade; é o erro barato, porque custa um reensaio. A detecção é a fração dos estouros efetivamente sinalizada.

A raiz do erro quadrático médio acompanha o erro médio, por penalizar mais os erros grandes. O viés, que é a média dos erros com sinal, indica se o modelo tende a subestimar, o que é o lado perigoso para uma triagem.

O coeficiente de determinação consta das tabelas, mas não decide nada: ele compara o erro à variação total dos dados, de modo que uma área de temperaturas pouco variáveis tem coeficiente baixo mesmo com previsões boas.

In [ ]:
MARGEM = 8.0


def metricas(y, pred, componentes):
    erro = pred - y
    lim = componentes.map(LIMITES).to_numpy(dtype=float)
    com_limite = ~np.isnan(lim)
    estoura = y[com_limite] >= lim[com_limite]
    alarme = pred[com_limite] + MARGEM >= lim[com_limite]
    return {"MAE": np.abs(erro).mean(),
            "RMSE": np.sqrt((erro ** 2).mean()),
            "R2": 1 - (erro ** 2).sum() / ((y - y.mean()) ** 2).sum(),
            "vies": erro.mean(),
            "aprov_indevidas": int((estoura & ~alarme).sum()),
            "reprov_desnecessarias": int((~estoura & alarme).sum()),
            "deteccao_%": 100 * (estoura & alarme).sum() / estoura.sum() if estoura.sum() else np.nan,
            "estouros": int(estoura.sum())}

## 4. Os modelos

A floresta aleatória constrói centenas de árvores de decisão independentes, cada uma treinada sobre uma amostra diferente dos dados e considerando um subconjunto das variáveis a cada divisão, e devolve a média das previsões. Os erros individuais das árvores, sendo em boa parte independentes, se cancelam na média (BREIMAN, 2001).

O gradiente constrói árvores rasas em sequência, cada uma treinada para corrigir o erro deixado pelas anteriores. Onde a floresta reduz variância pela média, o gradiente reduz viés pelo encadeamento (FRIEDMAN, 2001).

A mescla é a média aritmética das previsões dos dois. Como eles erram de formas diferentes, a média aproveita a complementaridade.

Nos três casos, a identidade do componente é substituída por duas estatísticas calculadas a partir das temperaturas dele: o patamar, que é a média, e a escala, que é o desvio-padrão. Essas estatísticas são derivadas do próprio alvo, e por isso um transformador as calcula dentro do pipeline, usando apenas as linhas de treino de cada dobra. O vazamento de informação do alvo fica impedido por construção. Este transformador foi desenvolvido no caderno de busca aleatória e é reaproveitado aqui.

In [ ]:
class EstatisticasComponente(BaseEstimator, TransformerMixin):
    """Substitui a identidade do componente por patamar e escala, ajustados so no treino."""

    def fit(self, X, y):
        y = pd.Series(np.asarray(y, dtype=float), index=X.index)
        grupo = y.groupby(X["componente"])
        # Mesma regra do caderno de modelagem: escala 1 quando o desvio nao pode ser calculado
        # (um unico valor no treino) ou quando e zero.
        self.patamar_, self.escala_ = grupo.mean(), grupo.std().fillna(1.0).replace(0, 1.0)
        self.patamar_geral_, self.escala_geral_ = float(y.mean()), float(y.std())
        return self

    def transform(self, X):
        comp = X["componente"]
        saida = X.drop(columns="componente").copy()
        saida["patamar"] = comp.map(self.patamar_).fillna(self.patamar_geral_).to_numpy()
        saida["escala"] = comp.map(self.escala_).fillna(self.escala_geral_).to_numpy()
        return saida


def pipeline(estimador):
    return Pipeline([("componente", EstatisticasComponente()), ("modelo", estimador)])


def floresta(**p):
    return pipeline(RandomForestRegressor(random_state=SEMENTE, n_jobs=1, **p))


def gradiente(**p):
    return pipeline(GradientBoostingRegressor(random_state=SEMENTE, **p))


def prever(fabrica, X, y, grupos):
    """Previsao de cada leitura quando a sua configuracao esta fora do treino."""
    pred = np.empty(len(y))
    for treino, teste in LeaveOneGroupOut().split(X, y, grupos):
        modelo = fabrica().fit(X.iloc[treino], y[treino])
        pred[teste] = modelo.predict(X.iloc[teste])
    return pred

## 5. Configuração fixa contra busca de hiperparâmetros

Hiperparâmetros são os valores de configuração do próprio algoritmo, tais como o número de árvores, a profundidade máxima e o número mínimo de exemplos por folha.

Cada algoritmo é avaliado em duas configurações. A configuração fixa é a do caderno de modelagem: trezentas árvores e todos os demais valores no padrão da biblioteca. A configuração buscada é a melhor entre quinze sorteadas por busca aleatória (BERGSTRA; BENGIO, 2012), com o mesmo espaço de busca do caderno de busca aleatória. O número de candidatos foi reduzido de quarenta para quinze para que a execução caiba numa sessão do Colab; com a mesma semente, os primeiros candidatos sorteados são os mesmos.

A busca usa a mesma validação agrupada por configuração, e escolhe pelo erro médio absoluto. Um detalhe de critério deve ser registrado: a busca escolhe os hiperparâmetros pela média do erro de cada dobra, e a comparação entre a configuração fixa e a buscada usa o erro de todas as leituras juntas. Como as dezessete dobras têm tamanhos diferentes, os dois critérios não coincidem exatamente, e a configuração buscada pode perder para a fixa na segunda medida mesmo tendo vencido na primeira. Para cada algoritmo e área, é mantida a melhor das duas configurações.

A mescla não é buscada separadamente: ela combina, em cada área, a floresta e o gradiente que venceram as suas respectivas comparações.

In [ ]:
N_CANDIDATOS = 15
ESPACO_ARVORE = {
    "modelo__n_estimators": randint(100, 601),
    "modelo__max_depth": [None, 4, 6, 8, 12, 16],
    "modelo__min_samples_leaf": randint(1, 11),
    "modelo__min_samples_split": randint(2, 11),
    "modelo__max_features": [1.0, 0.8, 0.6, 0.5, 0.33],
}
ESPACOS = {
    "floresta": {**ESPACO_ARVORE, "modelo__max_samples": [None, 0.6, 0.8]},
    "gradiente": {
        "modelo__n_estimators": randint(100, 801),
        "modelo__learning_rate": loguniform(0.01, 0.2),
        "modelo__max_depth": randint(2, 7),
        "modelo__min_samples_leaf": randint(1, 16),
        "modelo__subsample": uniform(0.5, 0.5),
        "modelo__max_features": [None, 0.8, 0.6],
        "modelo__loss": ["squared_error", "huber", "absolute_error"],
    },
}
FIXAS = {"floresta": lambda: floresta(n_estimators=300), "gradiente": lambda: gradiente(n_estimators=300)}
BASES = {"floresta": floresta, "gradiente": gradiente}

RESULTADO, ESCOLHA, PREVISAO = [], {}, {}
inicio = time.time()
for area in AREAS:
    X, y, g = DADOS[area]
    for alg in ("floresta", "gradiente"):
        pred_fixa = prever(FIXAS[alg], X, y, g)
        busca = RandomizedSearchCV(BASES[alg](), ESPACOS[alg], n_iter=N_CANDIDATOS,
                                   scoring="neg_mean_absolute_error", cv=LeaveOneGroupOut(),
                                   n_jobs=-1, random_state=SEMENTE, refit=False)
        busca.fit(X, y, groups=g)
        melhores = {k.replace("modelo__", ""): v for k, v in busca.best_params_.items()}
        pred_busca = prever(lambda: BASES[alg](**melhores), X, y, g)
        mae_fixa, mae_busca = np.abs(pred_fixa - y).mean(), np.abs(pred_busca - y).mean()
        venceu = "busca" if mae_busca < mae_fixa else "fixa"
        ESCOLHA[(area, alg)] = melhores if venceu == "busca" else {"n_estimators": 300}
        PREVISAO[(area, alg)] = pred_busca if venceu == "busca" else pred_fixa
        RESULTADO.append({"area": area, "algoritmo": alg, "MAE_fixa": mae_fixa,
                          "MAE_busca": mae_busca, "mantida": venceu})
    PREVISAO[(area, "mescla")] = (PREVISAO[(area, "floresta")] + PREVISAO[(area, "gradiente")]) / 2
    print(f"{area:<15} concluida | {(time.time() - inicio) / 60:.1f} min", flush=True)

TAB_BUSCA = pd.DataFrame(RESULTADO)
TAB_BUSCA["ganho"] = TAB_BUSCA["MAE_fixa"] - TAB_BUSCA["MAE_busca"]
print()
print(TAB_BUSCA.round(3).to_string(index=False))

### Hiperparâmetros mantidos por área

In [ ]:
def legivel(v):
    return round(float(v), 4) if isinstance(v, (float, np.floating)) else v


for (area, alg), p in ESCOLHA.items():
    print(f"{area:<15} {alg:<10}", {k: legivel(v) for k, v in p.items()})

## 6. Resultado por área

In [ ]:
NOMES = {"floresta": "Floresta", "gradiente": "Gradiente", "mescla": "Mescla RF + GB"}
linhas = []
for area in AREAS:
    X, y, g = DADOS[area]
    for alg in ("floresta", "gradiente", "mescla"):
        linhas.append({"area": area, "modelo": NOMES[alg],
                       **metricas(y, PREVISAO[(area, alg)], X["componente"])})
TAB_AREA = pd.DataFrame(linhas)

mae = TAB_AREA.pivot(index="area", columns="modelo", values="MAE").reindex(AREAS)[list(NOMES.values())]
mae["vencedor"] = mae.idxmin(axis=1)
print(mae.round(3).to_string())

In [ ]:
def agrega(modelo_por_area):
    """Metricas das sete areas juntas, com um modelo escolhido para cada uma."""
    y = np.concatenate([DADOS[a][1] for a in AREAS])
    pred = np.concatenate([PREVISAO[(a, modelo_por_area[a])] for a in AREAS])
    comp = pd.concat([DADOS[a][0]["componente"] for a in AREAS], ignore_index=True)
    return metricas(y, pred, comp)


INVERSO = {v: k for k, v in NOMES.items()}
agregado = {NOMES[alg]: agrega({a: alg for a in AREAS}) for alg in NOMES}
agregado["Vencedores por área"] = agrega({a: INVERSO[mae.loc[a, "vencedor"]] for a in AREAS})
TAB_AGREGADO = pd.DataFrame(agregado).T
print(TAB_AGREGADO.round(3).to_string())

## 7. O que foi escolhido, e por quê

A busca de hiperparâmetros teve efeitos opostos nos dois algoritmos, e o sinal foi o mesmo nas sete áreas.

Na floresta aleatória, a configuração fixa venceu a buscada em todas as áreas, com diferenças entre 0,059 e 1,137 grau. A floresta com os valores padrão da biblioteca já trabalha com árvores profundas e sem restrição, cujos erros se cancelam na média de trezentas árvores. O espaço de busca, além desses valores, só oferece formas de restringi-la, limitando a profundidade, exigindo mais exemplos por folha ou considerando menos variáveis por divisão, e restringir piorou. A varredura do caderno de modelagem havia chegado à mesma conclusão por outro caminho.

No gradiente, a configuração buscada venceu a fixa em todas as áreas, com ganhos entre 0,034 e 1,470 grau. O gradiente é sensível à taxa de aprendizado e à profundidade das árvores. As configurações vencedoras usam subamostragem das linhas em todas as áreas, profundidade entre três e cinco e número de árvores entre 147 e 661; nenhuma delas combina a taxa de um décimo com a profundidade três, que são os dois valores padrão, embora a profundidade três sozinha vença em três áreas. A mesma semente sorteia os mesmos quinze candidatos em todas as áreas, e por isso algumas áreas elegem a mesma configuração.

Com cada algoritmo na sua melhor configuração, o vencedor por área é o seguinte.

| Área | Vencedor | Erro do vencedor | Segundo colocado |
|---|---|---:|---|
| Oven HMI | Floresta | 2,393 | Mescla, 2,419 |
| CT Knobs | Gradiente | 2,769 | Mescla, 2,810 |
| Main Vent Sys | Gradiente | 3,868 | Mescla, 3,883 |
| IPC Cooling | Mescla | 3,822 | Gradiente, 3,829 |
| Mainback | Gradiente | 4,129 | Mescla, 4,257 |
| IPC Comp | Gradiente | 6,921 | Mescla, 7,056 |
| Coil Temps | Gradiente | 10,198 | Mescla, 10,524 |

O gradiente ajustado vence em cinco áreas. A floresta vence apenas em Oven HMI, que é a área de menor amplitude térmica, onde há pouca estrutura a ser explorada e a estabilidade da média de árvores independentes pesa mais. A mescla vence em IPC Cooling, por sete milésimos, o que equivale a um empate.

No agregado das sete áreas, o gradiente ajustado erra 5,954 graus, contra 6,474 da floresta e 6,075 da mescla. Usar em cada área o vencedor da comparação dá 5,950, praticamente o mesmo que usar o gradiente em todas, porque as duas áreas em que ele não vence são as de menor peso no total de leituras.

Uma ressalva acompanha esses números e não deve ser separada deles. O erro do gradiente ajustado é otimista, porque a configuração foi escolhida pela mesma validação que depois a avalia. O tamanho desse otimismo não foi medido para o gradiente; no caderno de modelagem, uma busca composta sobretudo por florestas mostrou que noventa por cento do ganho aparente era o próprio ato de escolher. Aqui, porém, há um indício de que parte do ganho é real: ele aparece nas sete áreas, com o mesmo sinal, e é grande onde a área é mais quente, chegando a 1,470 grau em Coil Temps. Um ganho produzido apenas pela escolha tenderia a ser pequeno e errático entre áreas.

## 8. Explicabilidade

A explicabilidade é apresentada para a floresta aleatória em IPC Comp, a área que concentra metade das leituras do projeto. O método é o de valores de Shapley aditivos (LUNDBERG; LEE, 2017), que atribui a cada variável a sua contribuição para cada previsão individual, de modo que a soma das contribuições reconstrói a previsão. Para árvores, esses valores são calculados de forma exata.

As contribuições são agrupadas por família de variável, para que a leitura seja feita em termos do problema, e não de colunas isoladas.

In [ ]:
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

AREA_SHAP = "ipc_comp"
X, y, g = DADOS[AREA_SHAP]
modelo = floresta(**ESCOLHA[(AREA_SHAP, "floresta")]).fit(X, y)
Xt = modelo.named_steps["componente"].transform(X)
valores = shap.TreeExplainer(modelo.named_steps["modelo"]).shap_values(Xt)

FAMILIAS = {"patamar": "Patamar do componente", "escala": "Escala do componente",
            "run_case_": "Tipo de ensaio", "losses_": "Perfil de perdas", "reason_": "Motivo do ensaio",
            "blower_direction_": "Direção do soprador", "housing_": "Compartimento",
            "board_position_": "Posição da placa", "deflector_hole_": "Furo do defletor",
            "material_coil_support_": "Suporte das bobinas"}


def familia(coluna):
    for prefixo, nome in FAMILIAS.items():
        if coluna == prefixo or coluna.startswith(prefixo):
            return nome
    return "Outra"


peso = pd.Series(np.abs(valores).mean(axis=0), index=Xt.columns).groupby(familia).sum()
peso = (100 * peso / peso.sum()).sort_values()
print((peso[::-1]).round(1).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
peso.plot.barh(ax=ax, color="#4a6fa5")
ax.set_xlabel("Participação na atribuição média, em %")
ax.set_title(f"Contribuição por família de variável, floresta em {AREA_SHAP}")
plt.tight_layout()
plt.show()

A leitura é coerente com a física da área. O patamar do componente, que informa em que ponto da placa de potência está o sensor, responde por um terço da explicação, o que é esperado: componentes diferentes operam em faixas de temperatura diferentes.

O tipo de ensaio e o perfil de perdas somam cerca de quarenta por cento. Ambos descrevem a carga elétrica imposta, e a placa de potência, que abriga transistores e capacitores, aquece principalmente pela potência que conduz. O motivo do ensaio, com 11,5 por cento, reúne as alterações de projeto que mudam o resfriamento, como a cobertura do dissipador e o caminho do ar.

Os atributos de geometria e montagem, como o furo do defletor, o compartimento, o suporte das bobinas, a direção do soprador e a posição da placa, somam menos de dez por cento. Nenhuma variável fisicamente implausível aparece entre as de maior peso.

Convém registrar o alcance desta leitura. Os valores explicam como o modelo chega às suas previsões, e não demonstram relação de causa no fogão. E o modelo explicado foi ajustado sobre todos os ensaios de desenvolvimento, e não dobra a dobra.

## 9. A formulação entregue

A formulação efetivamente entregue no caderno de modelagem parte desta comparação e acrescenta dois passos, ambos medidos naquele caderno.

O primeiro é a codificação do perfil de perdas escolhida por área, entre os treze indicadores detalhados e quatro categorias agrupadas. O segundo é um peso no ajuste que prioriza as leituras próximas do limite de cada componente, porque é ali que a decisão de triagem acontece. Em Main Vent Sys, a floresta dá lugar ao Extra Trees na média com o gradiente, troca identificada na comparação de algoritmos do fim da sprint e adotada apenas nessa região.

A célula abaixo reproduz a formulação entregue com a configuração fixa dos modelos, para comparação direta com as seções anteriores.

In [ ]:
REGRA = {"oven_hmi": ("floresta", "agrupado"), "ct_knobs": ("floresta", "detalhado"),
         "main_vent_sys": ("mescla ET + GB", "detalhado"), "ipc_cooling": ("floresta", "agrupado"),
         "mainback": ("floresta", "detalhado"), "ipc_comp": ("floresta", "agrupado"),
         "coil_temps": ("mescla", "detalhado")}


def agrupa_perdas(X):
    """Quatro niveis exclusivos no lugar dos treze indicadores de perdas."""
    perdas = [c for c in X.columns if c.startswith("losses_")]
    rotulo = lambda c: next((n for k, n in (("no losses", "none"), ("simone", "simone"),
                                              ("miro", "miro")) if k in c.lower()), "other")
    saida = X.drop(columns=perdas).copy()
    ativo = X[perdas].to_numpy(float)
    familias = np.array([rotulo(c) for c in perdas])
    for nivel in sorted(set(familias)):
        saida[f"perdas_{nivel}"] = (ativo[:, familias == nivel].sum(axis=1) > 0).astype(float)
    saida["perdas_nao_declarado"] = (ativo.sum(axis=1) == 0).astype(float)
    return saida


def peso_limite(y, componentes):
    """Peso tres sobre o limite, caindo para um a cerca de trinta graus dele."""
    lim = componentes.map(LIMITES).to_numpy(dtype=float)
    d = np.nan_to_num(np.abs(y - lim), nan=1e9)
    return np.where(np.isnan(lim), 1.0, 1.0 + 2.0 * np.exp(-(d / 15.0) ** 2))


def prever_entrega(area):
    alg, perdas = REGRA[area]
    X, y, g = DADOS[area]
    Xa = agrupa_perdas(X) if perdas == "agrupado" else X
    pred = np.empty(len(y))
    for treino, teste in LeaveOneGroupOut().split(Xa, y, g):
        w = peso_limite(y[treino], X["componente"].iloc[treino])
        # Main Vent Sys usa o Extra Trees no lugar da floresta, na mescla com o gradiente.
        base = (pipeline(ExtraTreesRegressor(n_estimators=300, random_state=SEMENTE, n_jobs=1))
                if alg == "mescla ET + GB" else floresta(n_estimators=300))
        p = base.fit(Xa.iloc[treino], y[treino], modelo__sample_weight=w).predict(Xa.iloc[teste])
        if alg in ("mescla", "mescla ET + GB"):
            gb = gradiente(n_estimators=300).fit(Xa.iloc[treino], y[treino], modelo__sample_weight=w)
            p = (p + gb.predict(Xa.iloc[teste])) / 2
        pred[teste] = p
    return pred


for area in AREAS:
    PREVISAO[(area, "entrega")] = prever_entrega(area)

comparacao = {"Vencedores por área": TAB_AGREGADO.loc["Vencedores por área"].to_dict(),
              "Formulação entregue": agrega({a: "entrega" for a in AREAS})}
TAB_ENTREGA = pd.DataFrame(comparacao).T
print(TAB_ENTREGA.round(3).to_string())

A formulação entregue erra 5,855 graus, contra 5,950 dos vencedores da comparação por área, e é superior também na triagem, com menos aprovações indevidas e menos reprovações desnecessárias. A raiz do erro quadrático e o coeficiente de determinação ficam praticamente iguais, 9,554 contra 9,535 e 0,898 contra 0,899, marginalmente a favor dos vencedores. O termo vencedor designa só o modelo de menor erro entre os três candidatos, com as perdas detalhadas, sem peso no treino e com a escolha avaliada pelos mesmos dados que a fizeram; não significa o melhor modelo possível.

| Indicador | Vencedores por área | Formulação entregue |
|---|---:|---:|
| Erro médio absoluto | 5,950 | 5,855 |
| Viés | +0,463 | +0,152 |
| Aprovações indevidas | 25 | 23 |
| Reprovações desnecessárias | 182 | 143 |
| Detecção | 87,0% | 88,1% |

O resultado merece atenção porque a formulação entregue usa os hiperparâmetros fixos, sem busca, e ainda assim supera a combinação dos modelos ajustados. A vantagem vem dos dois passos descritos no início desta seção. A codificação das perdas escolhida por área reduz o erro nas três áreas em que o perfil de perdas carrega pouco sinal. E o peso próximo ao limite concentra o ajuste onde a decisão de triagem acontece, o que se traduz em trinta e nove reprovações desnecessárias a menos e num viés três vezes menor.

Dois cuidados devem acompanhar esta comparação. O valor de 5,855 corresponde à regra por área escrita como constante, que foi derivada olhando os resultados. Com a escolha refeita dentro de cada dobra, o mesmo modelo mede entre 5,96 e 6,23 graus, conforme o método de aninhamento, e é essa faixa que o projeto reporta. O limite inferior vem do método usado ao longo da sprint, levemente otimista; o superior, de um método estrito, levemente pessimista. E o valor de 5,950 dos vencedores por área também é otimista, pela mesma razão, somada ao otimismo da busca de hiperparâmetros já discutido.

A leitura conjunta das seções 7 e 9 sugeria um próximo passo: combinar o gradiente ajustado com o peso e com a codificação das perdas por área. A hipótese foi testada no caderno de modelagem, com a escolha feita de forma aninhada, e não se sustentou. Pelo método de aninhamento usado ao longo da sprint, o ganho parecia ser de 0,107 grau; pelo método estrito, converteu-se numa perda de 0,448 grau. O gradiente ajustado continua melhor em Mainback e em Coil Temps, mas a validação não consegue identificar com segurança onde ele ajuda, e por isso a formulação entregue manteve o gradiente com os valores padrão.

### O que diferencia os vencedores da formulação entregue

Os vencedores da seção 7 e a formulação entregue não são o mesmo modelo medido de duas formas. Eles diferem em quatro pontos de construção e na forma de medir.

| Aspecto | Vencedores da comparação | Formulação entregue |
|---|---|---|
| Modelo por área | Gradiente em cinco áreas, floresta e mescla em uma cada | Floresta em cinco áreas; média com o gradiente em duas, com a floresta em Coil Temps e com o Extra Trees em Main Vent Sys |
| Hiperparâmetros | Buscados, no caso do gradiente | Fixos, com os valores padrão |
| Codificação das perdas | Treze indicadores detalhados em todas as áreas | Detalhada ou agrupada, conforme a área |
| Peso no treino | Igual para todas as leituras | Maior para as leituras próximas do limite |
| Forma de medir | Só sem a correção do viés de seleção | Sem a correção, nesta seção, e com a correção, no caderno de modelagem |
| Erro médio, sem a correção | 5,950 | 5,855 |
| Erro médio, com a correção | Não medido | De 5,96 a 6,23 |

Quando os dois são medidos da mesma forma, nesta seção, a formulação entregue erra menos e também tria melhor. Com a correção, só a formulação entregue foi medida; a peça central dos vencedores, o gradiente ajustado, foi testada dentro da entrega e piorou o erro, como descrito acima. O termo vencedor designa, portanto, o modelo de menor erro entre os três candidatos nas condições da comparação, e não o melhor modelo possível.

## 10. Métricas finais da formulação entregue e comparação com a Sprint 3

As métricas finais da formulação entregue vêm da Seção 23 do caderno de modelagem, e não deste caderno, porque é lá que a escolha por área é refeita dentro de cada dobra, com três sementes aleatórias. Os valores abaixo são as médias das três sementes e correspondem ao limite inferior da faixa, medido pelo método aninhado aproximado; por serem médias, as contagens de triagem podem ser fracionárias. As quatro primeiras áreas não têm leituras acima do limite no conjunto de desenvolvimento, e por isso não há detecção a calcular nelas.

| Área | MAE | RMSE | R² | Viés | Dentro de 10% | Aprovações indevidas | Reprovações desnecessárias | Detecção |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Oven HMI | 2,321 | 3,879 | 0,032 | +0,071 | 93,1% | 0 | 0 | sem excessos |
| CT Knobs | 3,051 | 5,265 | 0,539 | +0,333 | 85,8% | 0 | 0 | sem excessos |
| Main Vent Sys | 3,739 | 6,508 | 0,910 | −0,502 | 88,0% | 0 | 0 | sem excessos |
| IPC Cooling | 4,066 | 7,066 | 0,847 | +0,505 | 79,5% | 0 | 0 | sem excessos |
| Mainback | 4,586 | 8,167 | 0,896 | −0,138 | 79,4% | 6,0 | 13,3 | 76,9% |
| IPC Comp | 6,677 | 9,985 | 0,662 | +0,388 | 73,4% | 11,0 | 124,3 | 92,3% |
| Coil Temps | 10,612 | 15,641 | 0,883 | −0,113 | 69,0% | 6,3 | 19,3 | 74,7% |
| Agregado | 5,961 | 9,699 | 0,895 | +0,185 | 77,0% | 23,3 | 157,0 | 87,9% |

Pelo método estrito, medido no caderno analises_complementares, o agregado passa a 6,23, com 32,3 aprovações indevidas e detecção de 83,2%.

A tabela seguinte compara a formulação entregue com o modelo da Sprint 3, descrito na Seção 4.3 da documentação: um Random Forest por área, com hiperparâmetros fixos e as perdas reduzidas ao perfil. As duas medições usam a mesma validação por configuração, mas as bases não são idênticas, com 3.277 previsões na Sprint 3 e 3.232 nesta, e por isso as diferenças pequenas devem ser lidas com cautela.

| Área | MAE, Sprint 3 | MAE, Sprint 4 | Variação | Dentro de 10%, Sprint 3 | Dentro de 10%, Sprint 4 |
|---|---:|---:|---:|---:|---:|
| Oven HMI | 2,42 | 2,32 | −0,10 | 90,0% | 93,1% |
| CT Knobs | 3,16 | 3,05 | −0,11 | 84,4% | 85,8% |
| Main Vent Sys | 4,68 | 3,74 | −0,94 | 79,6% | 88,0% |
| IPC Cooling | 4,04 | 4,07 | +0,03 | 82,1% | 79,5% |
| Mainback | 5,68 | 4,59 | −1,09 | 71,8% | 79,4% |
| IPC Comp | 7,30 | 6,68 | −0,62 | 70,7% | 73,4% |
| Coil Temps | 11,71 | 10,61 | −1,10 | 65,4% | 69,0% |
| Agregado | 6,57 | 5,96 a 6,23 | −0,33 a −0,60 | 73,8% | 77,0% |

Na triagem com margem de oito graus, a Sprint 3 registrava 36 aprovações indevidas, 194 reprovações desnecessárias e detecção de 81,3%. A formulação entregue passa a 23,3 aprovações indevidas, 157,0 reprovações desnecessárias e 87,9% de detecção pelo método aproximado, e a 32,3 aprovações indevidas e 83,2% pelo estrito.

O erro cai em seis das sete áreas, com os maiores ganhos em Mainback, Coil Temps, Main Vent Sys e IPC Comp. Em IPC Cooling, a variação de três centésimos está dentro da oscilação esperada entre bases e sementes.

## 11. Ressalvas

A melhor configuração da busca é escolhida e avaliada pela mesma validação, e por isso o seu erro é otimista. Este efeito foi medido no caderno de modelagem com uma validação aninhada, em que a configuração é escolhida em cada dobra pelo erro nas demais e só então aplicada à dobra retida: o ganho aparente da busca era de 0,506 grau e o ganho aninhado foi de 0,051 grau, com intervalo de confiança que contém o zero (CAWLEY; TALBOT, 2010). Os ganhos da seção 5 devem ser lidos com essa proporção em mente. O mesmo efeito reapareceu, maior, ao testar o gradiente ajustado dentro da formulação entregue: a diferença entre o método de aninhamento aproximado e o estrito chegou a 0,555 grau.

Todos os números vêm de uma semente aleatória. A oscilação entre sementes foi medida nesta base e fica em centésimos de grau.

O conjunto de desenvolvimento tem quarenta e seis ensaios em dezessete configurações distintas, o que limita todas as conclusões deste caderno.

## Referências

BERGSTRA, James; BENGIO, Yoshua. Random search for hyper-parameter optimization. Journal of Machine Learning Research, v. 13, p. 281-305, 2012.

BREIMAN, Leo. Random forests. Machine Learning, v. 45, n. 1, p. 5-32, 2001.

CAWLEY, Gavin C.; TALBOT, Nicola L. C. On over-fitting in model selection and subsequent selection bias in performance evaluation. Journal of Machine Learning Research, v. 11, p. 2079-2107, 2010.

FRIEDMAN, Jerome H. Greedy function approximation: a gradient boosting machine. The Annals of Statistics, v. 29, n. 5, p. 1189-1232, 2001.

LUNDBERG, Scott M.; LEE, Su-In. A unified approach to interpreting model predictions. In: ADVANCES IN NEURAL INFORMATION PROCESSING SYSTEMS, 30., 2017, Long Beach. Proceedings [...]. Red Hook: Curran Associates, 2017. p. 4765-4774.

PEDREGOSA, Fabian et al. Scikit-learn: machine learning in Python. Journal of Machine Learning Research, v. 12, p. 2825-2830, 2011.